In [1]:
import os, shutil, glob, yaml, json, uuid, random, cv2
from pathlib import Path
import pandas as pd
from sklearn.model_selection import KFold

!pip install ultralytics
from ultralytics import YOLO
import albumentations as A

print(" Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 68.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 Setup complete


In [2]:
# ==== REQUIRED: your Roboflow YOLOv11 export URL ====
ROBOFLOW_EXPORT_URL = "https://app.roboflow.com/ds/CGhGDfCLRk?key=j6xhGogQ3e"
assert ROBOFLOW_EXPORT_URL.startswith("http"), "Paste a valid Roboflow YOLOv11 export URL."

# Training / K-Fold params
K = 5
EPOCHS = 2
IMGSZ = 640
BATCH = 32
MODEL = "yolov8n.pt"     # Changed from "yolov11n.pt" to "yolov8n.pt"
CONF_PRED = 0.25
PROJECT = "/content/runs"  # YOLO will save runs here

print(f"K={K}, EPOCHS={EPOCHS}, IMGSZ={IMGSZ}, BATCH={BATCH}, MODEL={MODEL}")

K=5, EPOCHS=2, IMGSZ=640, BATCH=32, MODEL=yolov8n.pt


In [ ]:
zip_path = "/content/dataset.zip"
!curl -L -o "$zip_path" "$ROBOFLOW_EXPORT_URL"

# Unzip into /content/data_raw
!mkdir -p /content/data_raw
!unzip -q -o "$zip_path" -d /content/data_raw

# Try to find a YAML (Roboflow includes one)
import glob
yaml_candidates = glob.glob("/content/data_raw/**/*.yaml", recursive=True)
yaml_candidates = [p for p in yaml_candidates if os.path.basename(p).lower() in ("data.yaml","dataset.yaml")]
BASE_YAML = yaml_candidates[0] if yaml_candidates else None
print("Found YAML:", BASE_YAML)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   904  100   904    0     0   2344      0 --:--:-- --:--:-- --:--:--  2341
100 1236M  100 1236M    0     0  81.8M      0  0:00:15  0:00:15 --:--:--  102M
Found YAML: /content/data_raw/data.yaml


In [ ]:
root = Path("/content/data_raw")
pool_img = Path("/content/pool/images"); pool_lbl = Path("/content/pool/labels")
pool_img.mkdir(parents=True, exist_ok=True); pool_lbl.mkdir(parents=True, exist_ok=True)

def copy_tree(img_dir, lbl_dir):
    if not img_dir.exists() or not lbl_dir.exists():
        return
    for img in img_dir.glob("*.*"):
        stem = img.stem
        lbl = lbl_dir / f"{stem}.txt"
        if lbl.exists():
            shutil.copy2(img, pool_img / img.name)
            shutil.copy2(lbl, pool_lbl / lbl.name)

# common layouts from Roboflow
cands = [
    ("train/images","train/labels"),
    ("valid/images","valid/labels"),
    ("val/images","val/labels"),
    ("test/images","test/labels"),
    ("images","labels")
]
for a,b in cands:
    copy_tree(root/a, root/b)

print("Pool:", len(list(pool_img.glob('*'))), "images,", len(list(pool_lbl.glob('*'))), "labels")
assert len(list(pool_img.glob('*'))) > 0, "No images found. Check your Roboflow URL."


Pool: 26574 images, 26574 labels


In [ ]:
if BASE_YAML and os.path.exists(BASE_YAML):
    y = yaml.safe_load(open(BASE_YAML))
    names = y.get("names")
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys(), key=lambda z: int(z))]
else:
    names = None

if not names:
    # fallback: infer roughly from one label
    first_lbl = next(pool_lbl.glob("*.txt"))
    max_id = 0
    with open(first_lbl) as f:
        for line in f:
            if line.strip():
                cid = int(line.split()[0]); max_id = max(max_id, cid)
    names = [f"class_{i}" for i in range(max_id+1)]

NC = len(names)
print("Classes (nc):", NC)
print("Names (first 12):", names[:12])


Classes (nc): 12
Names (first 12): ['c0 - Safe Driving', 'c1 - Texting', 'c2 - Talking on the phone', 'c3 - Operating the Radio', 'c4 - Drinking', 'c5 - Reaching Behind', 'c6 - Hair and Makeup', 'c7 - Talking to Passenger', 'd0 - Eyes Closed', 'd1 - Yawning', 'd2 - Nodding Off', 'd3 - Eyes Open']


In [ ]:
# Recompute exact counts by reading labels in /content/pool/labels
from pathlib import Path
from collections import Counter
pool_lbl = Path("/content/pool/labels")
cnt = Counter()
for p in pool_lbl.glob("*.txt"):
    for line in open(p, "r"):
        line = line.strip()
        if not line: continue
        parts = line.split()
        try:
            cid = int(float(parts[0]))
        except:
            continue
        cnt[cid] += 1

print("✅ Exact per-class object counts (labels across all files):")
for c in sorted(cnt.keys()):
    print(f"  Class {c}: {cnt[c]}")


✅ Exact per-class object counts (labels across all files):
  Class 0: 3538
  Class 1: 3000
  Class 2: 2988
  Class 3: 2963
  Class 4: 3007
  Class 5: 3035
  Class 6: 2990
  Class 7: 2777
  Class 8: 4395
  Class 9: 1007
  Class 10: 654
  Class 11: 14783


In [ ]:
# show a few augmented images that were created (fast manual sanity)
from IPython.display import display
from PIL import Image
import random, os
pool_img = Path("/content/pool/images")
candidates = list(pool_img.glob("*fastdup*.jpg")) + list(pool_img.glob("*_fastdup*.jpg")) + list(pool_img.glob("*_bal_*.jpg"))
random.shuffle(candidates)
sample = candidates[:12]
print("Showing up to 12 augmented images (if present):", len(sample))
for p in sample:
    display(Image.open(p).resize((480,320)))


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Continue targeted fast augmentation for remaining deficits
import random, cv2, shutil, time
from pathlib import Path
from collections import defaultdict, Counter

pool_img = Path("/content/pool/images")
pool_lbl = Path("/content/pool/labels")
TARGET_COUNT = 3000   # keep same target you used
MAX_DUP_PER_SRC = 6
random.seed(42)

# 1) build image->class list and class->image list (image-level)
img_to_classes = {}
class_to_imgs = defaultdict(list)
for lbl in pool_lbl.glob("*.txt"):
    stem = lbl.stem
    classes = set()
    with open(lbl) as f:
        for line in f:
            parts = line.strip().split()
            if not parts: continue
            try:
                cid = int(float(parts[0]))
            except:
                continue
            classes.add(cid)
    if classes:
        img_to_classes[stem] = classes
        for c in classes:
            class_to_imgs[c].append(stem)

# 2) compute how many images each class has (unique images)
class_image_counts = {c: len(set(v)) for c,v in class_to_imgs.items()}
print("Class image counts (samples):")
for c in sorted(class_image_counts.keys()):
    print(f"  Class {c}: {class_image_counts[c]} images")

# 3) find classes below target
targets = [c for c,ct in class_image_counts.items() if ct < TARGET_COUNT]
print("\nClasses still below target:", targets)
if not targets:
    print("All classes have >= TARGET_COUNT images. Nothing to do.")
else:
    created = 0
    start = time.time()
    for tgt in targets:
        need = TARGET_COUNT - class_image_counts[tgt]
        print(f"\n-- Augmenting class {tgt}: need +{need}")
        # prefer source images that contain the target class and the fewest other classes
        sources = sorted(class_to_imgs[tgt], key=lambda s: (len(img_to_classes.get(s,[])), random.random()))
        src_usage = Counter()
        i = 0
        attempts = 0
        while need > 0 and attempts < need * 8:
            attempts += 1
            if not sources:
                break
            src = random.choice(sources)
            if src_usage[src] >= MAX_DUP_PER_SRC:
                continue
            # check image file exists
            src_img = None
            for ext in (".jpg",".jpeg",".png"):
                p = pool_img / f"{src}{ext}"
                if p.exists():
                    src_img = p; break
            if src_img is None:
                continue
            src_lbl = pool_lbl / f"{src}.txt"
            if not src_lbl.exists():
                continue

            new_stem = f"{src}_tgt{tgt}_{src_usage[src]}"
            new_img = pool_img / f"{new_stem}.jpg"
            new_lbl = pool_lbl / f"{new_stem}.txt"

            # fast augment (flip + small brightness)
            do_flip = random.random() < 0.6
            bright = 1.0 + random.uniform(-0.10, 0.10)
            img = cv2.imread(str(src_img))
            if img is None:
                continue
            if do_flip:
                img = cv2.flip(img, 1)
            if bright != 1.0:
                img = cv2.convertScaleAbs(img, alpha=bright, beta=0)
            cv2.imwrite(str(new_img), img)
            shutil.copy2(str(src_lbl), str(new_lbl))

            # bookkeeping
            src_usage[src] += 1
            class_image_counts = {c: len(set(class_to_imgs[c])) for c in class_to_imgs.keys()}
            # also add new stem to index (for this run only)
            for c in img_to_classes.get(src, []):
                class_to_imgs[c].append(new_stem)
            created += 1
            need -= 1
            if created % 200 == 0:
                print(f"  created {created} images so far; elapsed {int(time.time()-start)}s")

    elapsed = int(time.time()-start)
    print(f"\nDone. Created {created} new images in {elapsed}s.")


Class image counts (samples):
  Class 0: 3538 images
  Class 1: 3000 images
  Class 2: 2988 images
  Class 3: 2963 images
  Class 4: 3007 images
  Class 5: 3035 images
  Class 6: 2990 images
  Class 7: 2777 images
  Class 8: 4386 images
  Class 9: 1007 images
  Class 10: 654 images
  Class 11: 14777 images

Classes still below target: [6, 2, 3, 9, 7, 10]

-- Augmenting class 6: need +10

-- Augmenting class 2: need +12

-- Augmenting class 3: need +37

-- Augmenting class 9: need +1993
  created 200 images so far; elapsed 1s
  created 400 images so far; elapsed 3s
  created 600 images so far; elapsed 6s
  created 800 images so far; elapsed 8s
  created 1000 images so far; elapsed 10s
  created 1200 images so far; elapsed 12s
  created 1400 images so far; elapsed 14s
  created 1600 images so far; elapsed 16s
  created 1800 images so far; elapsed 18s

-- Augmenting class 7: need +223
  created 2000 images so far; elapsed 21s

-- Augmenting class 10: need +2346
  created 2200 images so fa

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!zip -r /content/pool_backup.zip /content/pool
!cp /content/pool_backup.zip /content/drive/MyDrive/pool_backup.zip

Streaming output truncated to the last 5000 lines.
  adding: content/pool/images/img_6929_jpg.rf.a0019c5ef24bc525398b3a33c0317b32.jpg (deflated 1%)
  adding: content/pool/images/img_9499_jpg.rf.b31f56be66e8c8fc5c99d8ff758c20a7.jpg (deflated 0%)
  adding: content/pool/images/img_42048_jpg.rf.4739f9ea6237837d3be46ae27511dc28.jpg (deflated 0%)
  adding: content/pool/images/img_14634_jpg.rf.a24364e527690968ef6ddb153eb62a59.jpg (deflated 2%)
  adding: content/pool/images/img_22375_jpg.rf.9163e147e2b47e5b809473719245f0e1.jpg (deflated 2%)
  adding: content/pool/images/img_25688_jpg.rf.73b5952a059a6c3f12cd3c749ff3ceb0_tgt7_0.jpg (deflated 2%)
  adding: content/pool/images/img_42214_jpg.rf.a54452e4834c54115ea7e47f556e41e2.jpg (deflated 1%)
  adding: content/pool/images/img_3742_jpg.rf.3afb6826b23f9164e59cb07e0cb0f56c.jpg (deflated 1%)
  adding: content/pool/images/img_39747_jpg.rf.ebd806e91c980ca123d655dd4e2970fb.jpg (deflated 0%)
  adding: content/pool/images/img_24170_jpg.rf.b6d55dffa6a3f343

In [ ]:
pairs = []
for img in sorted(pool_img.glob("*.*")):
    lbl = pool_lbl / f"{img.stem}.txt"
    if lbl.exists():
        pairs.append((img, lbl))
print("Total image/label pairs:", len(pairs))
assert len(pairs) >= K, "Not enough images for chosen K."

fold_roots = []
kf = KFold(n_splits=K, shuffle=True, random_state=42)

for fold_idx, (_, val_idx) in enumerate(kf.split(pairs), start=1):
    base = Path(f"/content/folds/fold{fold_idx}")
    (base/"train/images").mkdir(parents=True, exist_ok=True)
    (base/"train/labels").mkdir(parents=True, exist_ok=True)
    (base/"val/images").mkdir(parents=True, exist_ok=True)
    (base/"val/labels").mkdir(parents=True, exist_ok=True)

    val_set = set(val_idx)

    for i, (img, lbl) in enumerate(pairs):
        split = "val" if i in val_set else "train"
        shutil.copy2(img,  base/f"{split}/images/{img.name}")
        shutil.copy2(lbl,  base/f"{split}/labels/{lbl.name}")

    data_yaml = {
        "train": str(base/"train/images"),
        "val":   str(base/"val/images"),
        "nc": NC,
        "names": names
    }
    with open(base/f"data_fold{fold_idx}.yaml", "w") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    fold_roots.append(base)
    print(f"Fold {fold_idx}:",
          len(list((base/'train/images').glob('*'))), "train,",
          len(list((base/'val/images').glob('*'))), "val")


Total image/label pairs: 30679
Fold 1: 24543 train, 6136 val
Fold 2: 24543 train, 6136 val
Fold 3: 24543 train, 6136 val
Fold 4: 24543 train, 6136 val
Fold 5: 24544 train, 6135 val


In [ ]:
# Zip the runs folder
!zip -r -q /content/runs_backup.zip /content/runs

# Copy to your Google Drive
!cp /content/runs_backup.zip /content/drive/MyDrive/runs_backup.zip

print("✅ runs folder zipped and saved to Google Drive!")

✅ runs folder zipped and saved to Google Drive!


In [ ]:
!cp /content/dataset.zip /content/drive/MyDrive/dataset.zip

In [ ]:
drive_pool_zip = "/content/drive/MyDrive/pool_backup.zip"
drive_folds_zip = "/content/drive/MyDrive/folds_backup.zip"
drive_dataset_zip = "/content/drive/MyDrive/dataset.zip"

local_pool_dir = "/content/pool"
local_folds_dir = "/content/folds"
local_dataset_dir = "/content/dataset"

print("📦 Extracting balanced pool dataset ...")
!unzip -q -o "$drive_pool_zip" -d /

print("📂 Extracting K-fold splits ...")
!unzip -q -o "$drive_folds_zip" -d /

import os
if os.path.exists(drive_dataset_zip):
    print("📁 Extracting base dataset ...")
    !mkdir -p /content/data_raw # Recreate the target directory
    !unzip -q -o "$drive_dataset_zip" -d /content/data_raw # Unzip into the specific directory
else:
    print("⚠️ No dataset.zip found — skipping.")

import os
print("\n Verification:")
# Use glob for robust counting, as os.listdir might fail if dir doesn't exist
print("Pool images:", len(list(Path(f"{local_pool_dir}/images").glob('*'))))
print("Pool labels:", len(list(Path(f"{local_pool_dir}/labels").glob('*'))))
print("Folds found:", os.listdir(local_folds_dir))

!head /content/folds/fold1/data_fold1.yaml

📦 Extracting balanced pool dataset ...
📂 Extracting K-fold splits ...
📁 Extracting base dataset ...

 Verification:
Pool images: 31021
Pool labels: 31021
Folds found: ['fold1', 'fold2', 'fold3', 'fold5', 'fold4']
train: /content/folds/fold1/train/images
val: /content/folds/fold1/val/images
nc: 12
names:
- c0 - Safe Driving
- c1 - Texting
- c2 - Talking on the phone
- c3 - Operating the Radio
- c4 - Drinking
- c5 - Reaching Behind


In [ ]:
from pathlib import Path

folds = sorted(Path("/content/folds").glob("fold*"))
print("📁 Found folds:", folds)

assert len(folds) == 5, f"Expected 5 folds, found {len(folds)}"


📁 Found folds: [PosixPath('/content/folds/fold1'), PosixPath('/content/folds/fold2'), PosixPath('/content/folds/fold3'), PosixPath('/content/folds/fold4'), PosixPath('/content/folds/fold5')]


In [ ]:
import shutil
from pathlib import Path

p = Path("/content/runs/y11_fold12")
if p.exists() and p.is_dir():
    shutil.rmtree(p)
    print("Deleted:", p)
else:
    print("Folder not found:", p)


Deleted: /content/runs/y11_fold12


In [ ]:
import subprocess
from pathlib import Path
from ultralytics import YOLO
import torch

results_csv_paths = []
best_weight_paths = []

# your configuration (keep as you defined earlier)
print("Using model:", MODEL)
print("Using epochs:", EPOCHS)
print("Using batch:", BATCH)
print("Using imgsz:", IMGSZ)

# device selection
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Training on device:", device)

# --- NEW: Ensure model weights are downloaded once ---
print(f"Attempting to load/download model weights: {MODEL}")
try:
    # Instantiating YOLO here will download weights if they don't exist
    _ = YOLO(MODEL)
    print(f"Successfully loaded/downloaded {MODEL}.")
except Exception as e:
    print(f"Error loading/downloading {MODEL}: {e}")
    raise RuntimeError(f"Failed to load/download model weights: {MODEL}. Please check network connectivity or permissions.")
# --- END NEW ---


# detect all fold directories
folds_list = sorted(Path("/content/folds").glob("fold*"))
print("Found folds:", folds_list)

for i, base in enumerate(folds_list, start=1):
    yaml_path = base / f"data_fold{i}.yaml"
    run_name = f"y11_fold{i}"

    # Using command line interface via subprocess to capture output
    cmd_list = [
        "yolo", "detect", "train",
        f"model={MODEL}",
        f"data={yaml_path}",
        f"epochs={EPOCHS}",
        f"imgsz={IMGSZ}",
        f"batch={BATCH}",
        f"device={device}",
        f"name={run_name}",
        f"project={PROJECT}",
        "hsv_h=0.015", "hsv_s=0.7", "hsv_v=0.4",
        "degrees=5", "translate=0.1", "scale=0.5", "shear=2", "perspective=0.0",
        "fliplr=0.5", "flipud=0.0", "mosaic=1.0", "mixup=0.1", "copy_paste=0.1"
    ]

    print(f"\n--------------------------------------")
    print(f"▶️ Training Fold {i}: {yaml_path}")
    print("--------------------------------------")
    print(f"Running command: {' '.join(cmd_list)}")

    # Execute the command and capture output
    process = subprocess.run(cmd_list, capture_output=True, text=True)

    # Reverted verbose printing for non-zero exit code
    if process.returncode != 0:
        raise RuntimeError(f"Training command returned non-zero exit code for fold {i}. Raw output: {process.stdout}\n{process.stderr}")

    # locate training output folder
    # Corrected path: YOLO saves directly under PROJECT/NAME, not PROJECT/detect/NAME
    run_dirs = sorted(Path(PROJECT).glob(f"{run_name}*"))

    # Reverted verbose printing for no run directory found
    if not run_dirs:
        raise RuntimeError(f"No run directory found for {run_name}. Training might have failed silently. Raw output: {process.stdout}\n{process.stderr}")

    run_dir = run_dirs[-1]
    csv_path = run_dir / "results.csv"
    best_path = run_dir / "weights" / "best.pt"

    results_csv_paths.append(str(csv_path))
    best_weight_paths.append(str(best_path))

    print(f"✔️ Fold {i} complete")
    print("Results CSV:", csv_path)
    print("Best Weights:", best_path)


Using model: yolov8n.pt
Using epochs: 2
Using batch: 32
Using imgsz: 640
Training on device: cuda
Attempting to load/download model weights: yolov8n.pt
Successfully loaded/downloaded yolov8n.pt.
Found folds: [PosixPath('/content/folds/fold1'), PosixPath('/content/folds/fold2'), PosixPath('/content/folds/fold3'), PosixPath('/content/folds/fold4'), PosixPath('/content/folds/fold5')]

--------------------------------------
▶️ Training Fold 1: /content/folds/fold1/data_fold1.yaml
--------------------------------------
Running command: yolo detect train model=yolov8n.pt data=/content/folds/fold1/data_fold1.yaml epochs=2 imgsz=640 batch=32 device=cuda name=y11_fold1 project=/content/runs hsv_h=0.015 hsv_s=0.7 hsv_v=0.4 degrees=5 translate=0.1 scale=0.5 shear=2 perspective=0.0 fliplr=0.5 flipud=0.0 mosaic=1.0 mixup=0.1 copy_paste=0.1
✔️ Fold 1 complete
Results CSV: /content/runs/y11_fold1/results.csv
Best Weights: /content/runs/y11_fold1/weights/best.pt

--------------------------------------

In [ ]:
rows = []
for csv_path in results_csv_paths:
    df = pd.read_csv(csv_path)
    last = df.iloc[-1]
    fold_name = Path(csv_path).parent.name
    rows.append({
        "fold": fold_name,
        "precision": last.get("metrics/precision(B)", float("nan")),
        "recall":    last.get("metrics/recall(B)", float("nan")),
        "mAP50":     last.get("metrics/mAP50(B)", float("nan")),
        "mAP50-95":  last.get("metrics/mAP50-95(B)", float("nan")),
        "val/box_loss": last.get("val/box_loss", float("nan")),
        "val/cls_loss": last.get("val/cls_loss", float("nan"))
    })

summary = pd.DataFrame(rows).set_index("fold")
summary.loc["mean"] = summary.mean(numeric_only=True)
summary


,precision,recall,mAP50,mAP50-95,val/box_loss,val/cls_loss
fold,,,,,,
y11_fold1,0.842490,0.848550,0.868340,0.65452,1.071010,0.90051
y11_fold2,0.842670,0.865350,0.870190,0.65029,1.092650,0.89636
y11_fold3,0.846530,0.871750,0.875270,0.65534,1.094120,0.89229
y11_fold4,0.827240,0.860020,0.872630,0.64794,1.084510,0.89679
y11_fold5,0.812080,0.880070,0.874350,0.62536,1.164350,0.88835
mean,0.834202,0.865148,0.872156,0.64669,1.101328,0.89486


In [ ]:
best_fold_name = summary.iloc[:-1]["mAP50"].idxmax()
best_idx = int(best_fold_name.split("fold")[-1])
BEST_WEIGHTS = str(best_weight_paths[best_idx - 1])

print("🏆 Best fold:", best_fold_name)
print("➡️ Using weights:", BEST_WEIGHTS)


🏆 Best fold: y11_fold3
➡️ Using weights: /content/runs/y11_fold3/weights/best.pt


In [ ]:
SAFE_BEST = "/content/best_yolo11.pt"
shutil.copy2(BEST_WEIGHTS, SAFE_BEST)
print("📌 Saved best weights to:", SAFE_BEST)

# zip the whole best run for download (optional)
best_run_dir = Path("/content/runs")/best_fold_name
ZIP_PATH = "/content/best_run_yolo11.zip"
shutil.make_archive("/content/best_run_yolo11", 'zip', best_run_dir)
print("📦 Zipped best run:", ZIP_PATH)

# OPTIONAL: save to Drive
from google.colab import drive
drive.mount('/content/drive')
shutil.copy2(SAFE_BEST, "/content/drive/MyDrive/best_yolo11.pt")
shutil.copy2(ZIP_PATH,  "/content/drive/MyDrive/best_run_yolo11.zip")


📌 Saved best weights to: /content/best_yolo11.pt
📦 Zipped best run: /content/best_run_yolo11.zip
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/best_run_yolo11.zip'

In [3]:
import gradio as gr
from PIL import Image
import uuid, os, json

SAFE_BEST1 = "/content/drive/MyDrive/tcs/best_yolo11.pt"
ui_model = YOLO(SAFE_BEST1)

OUTDIR = "/content/ui_outputs"
os.makedirs(OUTDIR, exist_ok=True)

DEFAULT_CONF = 0.25
IMGSZ = 640


def detect_and_json(image_path):
    uid = str(uuid.uuid4())[:8]
    ann_path  = os.path.join(OUTDIR, f"pred_{uid}.jpg")
    json_path = os.path.join(OUTDIR, f"pred_{uid}.json")

    results = ui_model.predict(
        source=image_path,
        conf=DEFAULT_CONF,
        imgsz=IMGSZ,
        save=False,
        verbose=False
    )

    dets = []
    for r in results:
        img_annot = Image.fromarray(r.plot()[:, :, ::-1])
        img_annot.save(ann_path)

        for b in r.boxes:
            cid = int(b.cls[0])
            cf  = float(b.conf[0])
            x1, y1, x2, y2 = b.xyxy[0].tolist()

            dets.append({
                "class_id": cid,
                "class_name": ui_model.names[cid],
                "confidence": round(cf, 4),
                "bbox_xyxy": [round(x1,2), round(y1,2), round(x2,2), round(y2,2)]
            })

    payload = {"image": os.path.basename(image_path), "detections": dets}
    with open(json_path, "w") as f:
        json.dump(payload, f, indent=2)

    return ann_path, payload, json_path


with gr.Blocks(title="Driver Distraction (YOLOv11) — Auto Detection") as demo:
    gr.Markdown("## 📷 Upload an image — detection runs automatically")

    img_input = gr.Image(type="filepath", label="Upload image (JPG/PNG)")

    img_out  = gr.Image(label="Annotated result", interactive=False)
    json_out = gr.JSON(label="Detections (JSON)")
    json_file = gr.File(label="Download JSON")

    # Auto-run when image changes
    img_input.change(
        detect_and_json,
        inputs=[img_input],
        outputs=[img_out, json_out, json_file]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://105bee7f805eeb2fdc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
